In [ ]:
import json
import logging
from pathlib import Path
from typing import Dict, Any, List
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

UN_MT_CSV = "/content/drive/MyDrive/25/Final/MT_Q/p1/01_build_keyword_maps_Output/data_master_text_clean_2.csv"
KEYWORD_MAP_JSON = "/content/drive/MyDrive/25/Final/MT_Q/p1/01_build_keyword_maps_Output/keyword_map.json"

ID_COL = "Sentence_ID"
EN_COL = "en"
REF_AR_COL = "ar"

MT_COLS = ["en_ar_DEEP", "en_ar_DEEPL", "en_ar_GOOGLE", "en_ar_TRANSFORMERS"]

OUT_ALL_CSV = "candidates_tiered_ALL.csv"
OUT_T1_CSV  = "candidates_TIER1.csv"
OUT_T2_CSV  = "candidates_TIER2.csv"

OUT_ALL_JSON = "candidates_tiered_ALL.json"
OUT_T1_JSON  = "candidates_TIER1.json"
OUT_T2_JSON  = "candidates_TIER2.json"

OUT_TIER_STATS_JSON = "tier_stats.json"
OUT_PATTERN_STATS_CSV = "pattern_stats.csv"
OUT_SUBTYPE_STATS_CSV = "subtype_stats.csv"

OMISSION_SENTINEL = "[OMITTED]"

TIER_CONFIDENCE = {"TIER1": 0.90, "TIER2": 0.40}

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("script2")

# ============================================================
# LOADERS
# ============================================================

def load_un_mt(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"UN_MT_CSV not found: {p}")

    df = pd.read_csv(p)
    required = [ID_COL, EN_COL, REF_AR_COL] + MT_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in UN_MT_CSV: {missing}\nFound: {df.columns.tolist()}")

    df[ID_COL] = df[ID_COL].astype(str)

    logger.info(f"Loaded UN+MT dataframe: rows={len(df)}")
    return df


def load_keyword_map(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"KEYWORD_MAP_JSON not found: {p}")

    with p.open("r", encoding="utf-8") as f:
        patterns = json.load(f)

    if not isinstance(patterns, list):
        raise ValueError("keyword_map.json must be a LIST of pattern dicts.")

    needed = {"sub_subtype", "keyword", "best_match"}
    bad = []
    for i, pat in enumerate(patterns):
        if not isinstance(pat, dict) or not needed.issubset(pat.keys()):
            bad.append(i)
    if bad:
        raise ValueError(f"keyword_map.json has invalid entries at indices: {bad[:20]}")

    logger.info(f"Loaded keyword_map patterns: {len(patterns)}")
    return patterns


# ============================================================
# CORE
# ============================================================

def build_candidates(df_un: pd.DataFrame, patterns: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Builds PER-TOOL rows.
    Core contract:
      - keyword must be in REF_AR_COL (strict)
      - Tier1 evidence is found ONLY in that tool's mt_output
      - Total Omission: keyword in REF, keyword missing from MT => Tier1
      - Otherwise Tier2
    """
    rows = []
    logger.info("Building candidates (per-tool rows; keyword-in-REF gate; MT evidence)...")

    for p in patterns:
        sub = str(p["sub_subtype"]).strip()
        keyword = str(p["keyword"]).strip()
        best_match = str(p["best_match"]).strip()
        is_total_omission = bool(p.get("is_total_omission", False))

        # ---------------------------
        # Pattern sanity guards
        # ---------------------------

        # never search literal [OMITTED] as keyword
        if not keyword or keyword == OMISSION_SENTINEL:
            continue

        # if NOT omission, best_match must be a real string
        if (not is_total_omission) and (not best_match or best_match == OMISSION_SENTINEL):
            continue

        # skip trivial patterns where "error" == "correct"
        if (not is_total_omission) and (best_match.strip() == keyword.strip()):
            continue

        for _, r in df_un.iterrows():
            ref_ar = str(r[REF_AR_COL] or "")
            if keyword not in ref_ar:
                continue  # gate: keyword must be in REF

            sentence_id = str(r[ID_COL])
            en_text = str(r[EN_COL] or "")

            for mt_col in MT_COLS:
                mt_text = str(r[mt_col] or "")

                # --- Tier assignment PER TOOL (mutually exclusive by if/else) ---
                if is_total_omission or best_match == OMISSION_SENTINEL:
                    # Evidence: REF has keyword AND MT missing keyword
                    if keyword not in mt_text:
                        tier = "TIER1"
                        signal = "OMISSION_EVIDENCE_REF_HAS_KW_MT_MISSING_KW"
                    else:
                        tier = "TIER2"
                        signal = "OMISSION_CONTEXT_REF_HAS_KW_MT_HAS_KW"
                else:
                    # Evidence: MT contains best_match (error form)
                    if best_match and (best_match in mt_text):
                        tier = "TIER1"
                        signal = "REF_HAS_KEYWORD_AND_MT_HAS_BESTMATCH"
                    else:
                        tier = "TIER2"
                        signal = "REF_HAS_KEYWORD_MT_MISSING_BESTMATCH"

                rows.append({
                    ID_COL: sentence_id,
                    EN_COL: en_text,
                    REF_AR_COL: ref_ar,

                    "MT Tool": mt_col,
                    "mt_output": mt_text,

                    "Sub-Subtype": sub,
                    "Keyword": keyword,
                    "Best_Match": best_match,
                    "is_total_omission": is_total_omission,

                    "tier": tier,
                    "retrieval_signal": signal,
                    "retrieval_confidence": TIER_CONFIDENCE[tier],
                })

    out = pd.DataFrame(rows)
    logger.info(f"Candidate rows (per-tool): {len(out)}")
    return out


# ============================================================
# VALIDATION + STATS
# ============================================================

def verify_tier_mutual_exclusivity(df: pd.DataFrame) -> None:
    """
    Must be mutually exclusive PER KEY.
    Key includes MT Tool because tiers are computed per tool output.
    """
    if df.empty:
        logger.warning("No rows to verify.")
        return

    KEY = [ID_COL, "MT Tool", "Sub-Subtype", "Keyword", "Best_Match"]
    tier_counts = df.groupby(KEY)["tier"].nunique().reset_index(name="tier_count")
    conflicts = tier_counts[tier_counts["tier_count"] > 1]

    if len(conflicts) > 0:
        conflicts.to_csv("tier_conflicts.csv", index=False, encoding="utf-8-sig")
        raise ValueError(
            f"Tier exclusivity violated: {len(conflicts)} conflicts saved to tier_conflicts.csv"
        )

    logger.info("✅ Tier mutual exclusivity verified: 0 conflicts.")


def compute_stats(df: pd.DataFrame) -> Dict[str, Any]:
    if df.empty:
        return {"total_rows": 0}

    return {
        "total_rows": int(len(df)),
        "tier_row_counts": df["tier"].value_counts().to_dict(),
        "tier_unique_sentence_counts": df.groupby("tier")[ID_COL].nunique().to_dict(),
        "unique_sentence_ids": int(df[ID_COL].nunique()),
        "unique_patterns": int(df[["Sub-Subtype", "Keyword", "Best_Match"]].drop_duplicates().shape[0]),
        "unique_tools": int(df["MT Tool"].nunique()),
    }


def pattern_stats(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(
            columns=["Sub-Subtype", "Keyword", "Best_Match", "tier", "n_rows", "n_unique_sentence_ids"]
        )

    g = df.groupby(
        ["Sub-Subtype", "Keyword", "Best_Match", "tier"],
        as_index=False
    ).agg(
        n_rows=(ID_COL, "count"),
        n_unique_sentence_ids=(ID_COL, "nunique"),
    )

    return g.sort_values(["Sub-Subtype", "tier", "n_rows"], ascending=[True, True, False])


def subtype_stats(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(
            columns=["Sub-Subtype", "tier", "n_rows", "n_unique_sentence_ids"]
        )

    g = df.groupby(
        ["Sub-Subtype", "tier"],
        as_index=False
    ).agg(
        n_rows=(ID_COL, "count"),
        n_unique_sentence_ids=(ID_COL, "nunique"),
    )

    return g.sort_values(["Sub-Subtype", "tier"])


def save_json_records(df: pd.DataFrame, out_path: str) -> None:
    """
    Saves as list-of-dicts JSON (simple, Script3 can group as needed).
    """
    Path(out_path).write_text(
        json.dumps(df.to_dict(orient="records"), ensure_ascii=False, indent=2),
        encoding="utf-8"
    )


# ============================================================
# MAIN
# ============================================================

def main():
    logger.info("=" * 80)
    logger.info("SCRIPT 2 — Tiered Candidate Extraction (keyword in REF only; MT evidence per tool)")
    logger.info("=" * 80)

    df_un = load_un_mt(UN_MT_CSV)
    patterns = load_keyword_map(KEYWORD_MAP_JSON)

    df_candidates = build_candidates(df_un, patterns)

    # Hard guarantee: mutual exclusivity (per-tool key)
    verify_tier_mutual_exclusivity(df_candidates)

    # Save ALL
    df_candidates.to_csv(OUT_ALL_CSV, index=False, encoding="utf-8-sig")
    save_json_records(df_candidates, OUT_ALL_JSON)
    logger.info(f"Saved: {OUT_ALL_CSV} and {OUT_ALL_JSON}")

    # Save TIER1 and TIER2
    df_t1 = df_candidates[df_candidates["tier"] == "TIER1"].copy()
    df_t2 = df_candidates[df_candidates["tier"] == "TIER2"].copy()

    df_t1.to_csv(OUT_T1_CSV, index=False, encoding="utf-8-sig")
    save_json_records(df_t1, OUT_T1_JSON)

    df_t2.to_csv(OUT_T2_CSV, index=False, encoding="utf-8-sig")
    save_json_records(df_t2, OUT_T2_JSON)

    logger.info(f"Saved: {OUT_T1_CSV}/{OUT_T1_JSON} (rows={len(df_t1)})")
    logger.info(f"Saved: {OUT_T2_CSV}/{OUT_T2_JSON} (rows={len(df_t2)})")

    # Stats
    stats = compute_stats(df_candidates)
    Path(OUT_TIER_STATS_JSON).write_text(
        json.dumps(stats, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    logger.info(f"Saved: {OUT_TIER_STATS_JSON}")
    logger.info(f"Tier distribution: {stats.get('tier_row_counts', {})}")

    # Pattern + subtype stats
    pattern_stats(df_candidates).to_csv(
        OUT_PATTERN_STATS_CSV, index=False, encoding="utf-8-sig"
    )
    subtype_stats(df_candidates).to_csv(
        OUT_SUBTYPE_STATS_CSV, index=False, encoding="utf-8-sig"
    )

    logger.info(f"Saved: {OUT_PATTERN_STATS_CSV}")
    logger.info(f"Saved: {OUT_SUBTYPE_STATS_CSV}")

    logger.info("DONE ✅")


if __name__ == "__main__":
    main()


In [ ]:
# =============================================================================
# CELL X — Script 2 Tier Statistics (CSV exports) — ALIGNED & FIXED
# =============================================================================
import pandas as pd
from pathlib import Path

print("=" * 80)
print("CELL X: Tier Statistics Exports (Script-2 aligned)")
print("=" * 80)

# ---------------------------------------------------------------------------
# 0) Load candidates dataframe (Script 2 contract)
# ---------------------------------------------------------------------------
# Preferred: df_candidates from Script 2
if "df_candidates" in globals() and df_candidates is not None and len(df_candidates) > 0:
    df = df_candidates.copy()
    print(f"✅ Using df_candidates from memory (rows={len(df)})")
else:
    # Fallback: reload Script 2 output
    OUT_ALL_CSV = globals().get("OUT_ALL_CSV", "candidates_tiered_ALL.csv")
    path = Path(OUT_ALL_CSV)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find df_candidates in memory and CSV not found: {path}"
        )
    print(f"⚠️ Reloading candidates from: {path}")
    df = pd.read_csv(path, encoding="utf-8-sig")

# Output directory (current dir by default)
out_dir = Path(".")
out_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 1) Required Script-2 columns (STRICT)
# ---------------------------------------------------------------------------
REQUIRED_COLS = [
    "Sentence_ID",
    "tier",
    "Sub-Subtype",
    "Keyword",
    "Best_Match",
    "MT Tool",
]

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing required Script-2 columns: {missing}\n"
        f"Found columns: {df.columns.tolist()}"
    )

# Normalize dtypes
df["Sentence_ID"] = df["Sentence_ID"].astype(str)
df["Sub-Subtype"] = df["Sub-Subtype"].astype(str)
df["tier"] = df["tier"].astype(str)

# ---------------------------------------------------------------------------
# Output paths
# ---------------------------------------------------------------------------
OUT_TIER_OVERALL_CSV    = out_dir / "tier_summary_overall.csv"
OUT_TIER_BY_SUBTYPE_CSV = out_dir / "tier_summary_by_subtype.csv"
OUT_TIER_BY_TOOL_CSV    = out_dir / "tier_summary_by_tool.csv"
OUT_SMALL_POOLS_CSV     = out_dir / "tier_small_pools.csv"
OUT_TIER_CONFLICTS_CSV  = out_dir / "tier_conflicts.csv"

# ---------------------------------------------------------------------------
# 2) Overall tier stats
# ---------------------------------------------------------------------------
df["_pattern_key"] = (
    df[["Sub-Subtype", "Keyword", "Best_Match", "is_total_omission"]]
    .astype(str)
    .agg("||".join, axis=1)
)

overall = (
    df.groupby("tier")
    .agg(
        rows=("Sentence_ID", "size"),
        unique_sentence_ids=("Sentence_ID", "nunique"),
        unique_patterns=("_pattern_key", "nunique"),
    )
    .reset_index()
    .sort_values("tier")
)

overall.to_csv(OUT_TIER_OVERALL_CSV, index=False, encoding="utf-8-sig")

# ---------------------------------------------------------------------------
# 3) Tier × Sub-Subtype breakdown
# ---------------------------------------------------------------------------
tier_order = ["TIER1", "TIER2"]
present = [t for t in tier_order if t in df["tier"].unique()]
df["tier"] = pd.Categorical(df["tier"], categories=present, ordered=True)

by_subtype = (
    df.groupby(["Sub-Subtype", "tier"])
    .agg(
        rows=("Sentence_ID", "size"),
        unique_sentence_ids=("Sentence_ID", "nunique"),
    )
    .reset_index()
)

pivot_rows = by_subtype.pivot_table(
    index="Sub-Subtype", columns="tier", values="rows", fill_value=0
).reset_index()

pivot_uniq = by_subtype.pivot_table(
    index="Sub-Subtype", columns="tier", values="unique_sentence_ids", fill_value=0
).reset_index()

pivot_rows.columns = [
    c if c == "Sub-Subtype" else f"rows_{c}" for c in pivot_rows.columns
]
pivot_uniq.columns = [
    c if c == "Sub-Subtype" else f"uniqSID_{c}" for c in pivot_uniq.columns
]

by_subtype_summary = pivot_rows.merge(pivot_uniq, on="Sub-Subtype", how="outer").fillna(0)

row_cols = [c for c in by_subtype_summary.columns if c.startswith("rows_")]
uniq_cols = [c for c in by_subtype_summary.columns if c.startswith("uniqSID_")]

by_subtype_summary["rows_TOTAL"] = by_subtype_summary[row_cols].sum(axis=1).astype(int)
by_subtype_summary["uniqSID_TOTAL"] = by_subtype_summary[uniq_cols].sum(axis=1).astype(int)

by_subtype_summary = by_subtype_summary.sort_values("rows_TOTAL", ascending=False)
by_subtype_summary.to_csv(OUT_TIER_BY_SUBTYPE_CSV, index=False, encoding="utf-8-sig")

# ---------------------------------------------------------------------------
# 4) Tier × MT Tool breakdown
# ---------------------------------------------------------------------------
by_tool = (
    df.groupby(["MT Tool", "tier"])
    .agg(
        rows=("Sentence_ID", "size"),
        unique_sentence_ids=("Sentence_ID", "nunique"),
    )
    .reset_index()
    .sort_values(["MT Tool", "tier"])
)

by_tool.to_csv(OUT_TIER_BY_TOOL_CSV, index=False, encoding="utf-8-sig")

# ---------------------------------------------------------------------------
# 5) Small pool detector
# ---------------------------------------------------------------------------
for col in ["rows_TIER1", "rows_TIER2"]:
    if col not in by_subtype_summary.columns:
        by_subtype_summary[col] = 0

small = by_subtype_summary.copy()
small["flag_low_TIER1"] = small["rows_TIER1"] < 10
small["flag_low_TOTAL"] = small["rows_TOTAL"] < 50

small = small.sort_values(
    ["flag_low_TIER1", "rows_TIER1", "rows_TOTAL"],
    ascending=[False, True, True],
)

small.to_csv(OUT_SMALL_POOLS_CSV, index=False, encoding="utf-8-sig")

# ---------------------------------------------------------------------------
# 6) Tier mutual exclusivity check (MATCHES SCRIPT 2)
# ---------------------------------------------------------------------------
KEY = ["Sentence_ID", "MT Tool", "Sub-Subtype", "Keyword", "Best_Match"]

tier_conflicts = (
    df.groupby(KEY)["tier"].nunique().reset_index(name="tier_count")
)
conflicts = tier_conflicts[tier_conflicts["tier_count"] > 1]

if len(conflicts) > 0:
    conflicts.to_csv(OUT_TIER_CONFLICTS_CSV, index=False, encoding="utf-8-sig")
    raise ValueError(
        f"Tier exclusivity violated: {len(conflicts)} conflicts "
        f"(saved to {OUT_TIER_CONFLICTS_CSV})"
    )
else:
    print("✅ Tier mutual exclusivity verified: 0 conflicts.")

# ---------------------------------------------------------------------------
# 7) Quick summary
# ---------------------------------------------------------------------------
print("\n✅ Tier stats written to:", out_dir.resolve())
print("Overall rows per tier:", df["tier"].value_counts().to_dict())
print("Unique Sentence_IDs:", df["Sentence_ID"].nunique())
print("Sub-Subtype count:", df["Sub-Subtype"].nunique())

print("\nTop 10 Sub-Subtypes by total rows:")
print(
    by_subtype_summary[["Sub-Subtype", "rows_TOTAL"]]
    .head(10)
    .to_string(index=False)
)


In [ ]:
import pandas as pd

CANDIDATES_PATH = "/content/candidates_tiered_ALL.csv"  # adjust if needed

df_candidates = pd.read_csv(CANDIDATES_PATH)

print("Rows:", len(df_candidates))
print("Unique Sentence_ID:", df_candidates["Sentence_ID"].nunique())
print("Tier counts:")
print(df_candidates["tier"].value_counts())


In [ ]:
dup = (
    df_candidates
    .groupby(
        ['Sentence_ID', 'Sub-Subtype', 'Keyword', 'Best_Match']
    )['tier']
    .nunique()
    .reset_index(name="tier_count")
)

violations = dup[dup["tier_count"] > 1]

print(f"Tier conflicts found: {len(violations)}")

if len(violations) > 0:
    print("❌ BUG: Same sentence–pattern assigned to multiple tiers")
    display(violations.head(10))
else:
    print("✅ OK: TIER1 and TIER2 are mutually exclusive")


In [ ]:
# --- Tier mutual exclusivity audit (should be ZERO) ---

KEY = ['Sentence_ID', 'MT Tool', 'Sub-Subtype', 'Keyword', 'Best_Match']

tier_counts = (
    df_candidates
    .groupby(KEY)['tier']
    .nunique()
    .reset_index(name='tier_count')
)

conflicts = tier_counts[tier_counts['tier_count'] > 1]

print("Conflicts:", len(conflicts))

if len(conflicts) > 0:
    conflicts.to_csv("tier_conflicts.csv", index=False, encoding="utf-8-sig")
    raise ValueError("Tier mutual exclusivity violated. See tier_conflicts.csv")
else:
    print("✅ Mutual exclusivity verified: no candidate appears in multiple tiers.")
